## Data Generation

This notebook generates a synthetic loan dataset used for pricing and scenario analysis in subsequent notebooks.

In [145]:
import numpy as np
import pandas as pd



np.random.seed(42)
n = 1500000

# -----------------------------
# 1. Basic borrower / loan data
# -----------------------------
fico = np.random.randint(620, 801, n)
ltv = np.clip(np.random.normal(78, 10, n), 50, 97).round(1)
loan_amount = np.clip(np.random.normal(450000, 175000, n), 100000, 1500000).round(0)

purpose = np.random.choice(
    ["purchase", "refinance", "cash_out"],
    size=n,
    p=[0.55, 0.30, 0.15]
)

occupancy = np.random.choice(
    ["primary", "second_home", "investment"],
    size=n,
    p=[0.75, 0.10, 0.15]
)

market_volatility = np.random.choice(
    ["low", "medium", "high"],
    size=n,
    p=[0.45, 0.35, 0.20]
)

# -----------------------------
# 2. Rate generation
# -----------------------------
# prevailing market rate
market_rate = np.clip(np.random.normal(6.50, 0.75, n), 4.75, 8.50)
market_rate = np.round(market_rate / 0.125) * 0.125
market_rate = np.round(market_rate, 3)

# offered rate relative to market
# negative values = better than market for borrower
rate_adjustment = np.random.choice(
    [-0.375, -0.250, -0.125, 0.000, 0.125, 0.250],
    size=n,
    p=[0.05, 0.10, 0.20, 0.40, 0.15, 0.10]
)

offered_rate = market_rate + rate_adjustment
offered_rate = np.clip(offered_rate, 4.75, 8.50)
offered_rate = np.round(offered_rate / 0.125) * 0.125
offered_rate = np.round(offered_rate, 3)

rate_diff = offered_rate - market_rate
rate_cut_25bps = (rate_diff <= -0.25).astype(int)

# -----------------------------
# 3. Build lock probability
# -----------------------------
# Start with a baseline log-odds
logit = -0.50

# Better offered rate than market should increase lock probability
# negative rate_diff is better for the borrower, so subtract it
#logit += -4.0 * rate_diff
logit += -6.0 * rate_diff

# Purpose effects
logit += np.where(purpose == "purchase", 0.45, 0)
logit += np.where(purpose == "refinance", 0.10, 0)
logit += np.where(purpose == "cash_out", -0.25, 0)

# Occupancy effects
logit += np.where(occupancy == "primary", 0.15, 0)
logit += np.where(occupancy == "second_home", -0.10, 0)
logit += np.where(occupancy == "investment", -0.30, 0)

# Credit / leverage effects
logit += 0.003 * (fico - 700)          # higher fico slightly better
logit += -0.015 * (ltv - 80)           # higher ltv slightly worse
logit += -0.000001 * (loan_amount - 450000)  # larger loans slightly less likely

# Market environment
logit += np.where(market_volatility == "low", 0.20, 0)
logit += np.where(market_volatility == "medium", 0.00, 0)
logit += np.where(market_volatility == "high", -0.35, 0)

# Extra treatment heterogeneity:
# refinances are more sensitive to better rates
logit += np.where((purpose == "refinance") & (rate_diff < 0), 0.35, 0)

# Convert log-odds to probability
lock_prob = 1 / (1 + np.exp(-logit))

# Keep probabilities in a sensible range
lock_prob = np.clip(lock_prob, 0.02, 0.98)

# -----------------------------
# 4. Simulate outcome
# -----------------------------
locked = np.random.binomial(1, lock_prob, n)

# -----------------------------
# 5. Final dataframe
# -----------------------------
df = pd.DataFrame({
    "fico": fico,
    "ltv": ltv,
    "loan_amount": loan_amount,
    "purpose": purpose,
    "occupancy": occupancy,
    "market_volatility": market_volatility,
    "market_rate": market_rate,
    "offered_rate": offered_rate,
    "rate_diff": rate_diff,
    "rate_cut_25bps": rate_cut_25bps,
    "lock_prob": lock_prob,
    "locked": locked
})

print(df.head())
print("\nOverall lock rate:", df["locked"].mean().round(4))
print("\nLock rate by rate diff:")
print(df.groupby("rate_diff")["locked"].mean().sort_index())



   fico    ltv  loan_amount    purpose   occupancy market_volatility  \
0   722 97.000  491,983.000  refinance     primary              high   
1   799 86.000  451,628.000  refinance     primary               low   
2   712 78.600  753,610.000  refinance  investment               low   
3   634 54.300  345,063.000   cash_out  investment               low   
4   726 66.800  756,271.000   purchase  investment               low   

   market_rate  offered_rate  rate_diff  rate_cut_25bps  lock_prob  locked  
0        6.250         6.250      0.000               0      0.303       0  
1        6.375         6.500      0.125               0      0.356       0  
2        6.125         6.125      0.000               0      0.322       0  
3        5.750         5.750      0.000               0      0.364       1  
4        7.375         7.250     -0.125               0      0.639       1  

Overall lock rate: 0.5091

Lock rate by rate diff:
rate_diff
-0.375   0.895
-0.250   0.807
-0.125   0.66

In [146]:
df["fico_bucket"] = pd.cut(
    df["fico"],
    bins=[600, 660, 700, 740, 780, 850],
    labels=["600-659", "660-699", "700-739", "740-779", "780+"]
)

df["ltv_bucket"] = pd.cut(
    df["ltv"],
    bins=[0, 60, 70, 80, 90, 100],
    labels=["<60", "60-70", "70-80", "80-90", "90+"]
)

In [147]:
df.to_parquet("mortgage_data.parquet", index=False)

The generated dataset will be used in Notebook 2 for scenario simulation and analysis.